# RAG Pipeline using Amazon Bedrock (Nova Micro + Titan Embeddings)

End-to-end RAG using **ChatBedrockConverse** (messages-based) for Amazon Nova models.

## 1. Imports

In [30]:
import boto3
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import BedrockEmbeddings

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage

## 2. Configuration

In [31]:

REGION = "us-east-1"
PDF_PATH = "../data/pdf/attention.pdf"
INDEX_DIR = "faiss_index"
TOP_K = 4

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)


## 3. Embeddings – Amazon Titan Text Embeddings v2

In [32]:

embeddings = BedrockEmbeddings(
    client=bedrock_runtime,
    model_id="amazon.titan-embed-text-v2:0"
)


## 4. LLM – Amazon Nova Micro (ChatBedrockConverse)

In [33]:
llm = ChatBedrockConverse(
    client=bedrock_runtime,
    model="amazon.nova-micro-v1:0",
    temperature=0.2,
    max_tokens=512,
)


## 5. Helper Functions (RAG Prompt + Ask)

In [34]:

def build_rag_prompt(question: str, context_docs) -> str:
    context_text = "\n".join(d.page_content for d in context_docs)
    return (
        "You are a helpful assistant.\n"
        "Answer the user ONLY using the provided context.\n"
        "If the answer is not present in the context, say: \"I don't know based on the provided document.\"\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def ask_rag(question: str, retriever) -> str:
    docs = retriever.invoke(question)
    prompt = build_rag_prompt(question, docs)
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


## 6. Load PDF and Split into Chunks

In [35]:
pdf_path = Path(PDF_PATH)
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path.resolve()}")

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

len(chunks)

52

## 7. Build & Persist FAISS Vector Store

In [36]:
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(INDEX_DIR)

## 8. Load Vector Store and Ask Question

In [37]:

vector_store = FAISS.load_local(
    INDEX_DIR,
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K},
)

question = "Can you briefly describe the context of attention.pdf?"
answer = ask_rag(question, retriever)
print(answer)


The context describes an attention function, which maps a query and a set of key-value pairs to an output, where all are vectors. The output is computed as a weighted sum of the values, where the weights are determined by a compatibility function between the query and the keys. The document discusses two common types of attention: additive attention and dot-product (multiplicative) attention. It highlights that dot-product attention is faster and more space-efficient, though additive attention can outperform it for larger dimensions. The document also introduces "Scaled Dot-Product Attention," which scales the dot products by 1/√dk before applying a softmax function to obtain the weights on the values.
